# pyasc编程范式

## 概述

本节介绍pyasc的编程模型和核心编程范式。pyasc是Ascend C的Python编程接口，通过JIT编译将Python代码转换为NPU可执行Kernel。我们将从Host-Device异构协同、SPMD并行模型、`@asc.jit`装饰器三个维度建立对pyasc编程范式的整体认知。

### 学习目标

完成本节后，开发者应能够：

1. 理解Host-Device异构协同机制和各自职责；
2. 理解SPMD并行编程模型和`asc.get_block_idx()`的作用；
3. 掌握`@asc.jit`装饰器的使用和编译/运行参数配置；
4. 了解pyasc的完整编译链路（Python→AST→ASC-IR→Ascend C→Kernel二进制）。


In [ ]:
# 环境初始化
!mkdir -p Sources/02.02

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. Host-Device异构协同机制

昇腾NPU计算采用**异构计算架构**，计算任务在两类处理器上协同完成：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">角色</th>
      <th align="left">处理器</th>
      <th align="left">核心职责</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Host侧</td>
      <td>CPU</td>
      <td>数据准备、内存分配、核函数下发（Launch）、结果验证</td>
    </tr>
    <tr>
      <td>Device侧</td>
      <td>AI Core（NPU）</td>
      <td>执行<code>@asc.jit</code>修饰的核函数，完成数据搬运和矢量/矩阵计算</td>
    </tr>
  </tbody>
</table>

核函数是Host侧和Device侧连接的桥梁：Host侧通过**内核调用符**`kernel[核数, stream]()`下发核函数，Device侧多个核并行执行同一核函数处理不同数据分片。

<img src="./images/host_device_arch.png" alt="Host-Device异构架构" width="700px">


---
# 2. SPMD并行编程模型

pyasc采用**SPMD（Single Program Multiple Data，单程序多数据）**并行编程模型。所有核执行同一份核函数代码，但每个核通过`asc.get_block_idx()`获取自己的编号，计算数据偏移，处理数据的不同分片。

以Add算子为例，假设共启用8个核，数据整体长度为8x2048个元素：

<img src="./images/spmd_parallel_model.png" alt="SPMD多核并行模型" width="700px">

每个核的处理逻辑如下：

```text
核0: offset = 0 * 2048 = 0,      处理元素 [0, 2048)
核1: offset = 1 * 2048 = 2048,   处理元素 [2048, 4096)
核2: offset = 2 * 2048 = 4096,   处理元素 [4096, 6144)
...
核7: offset = 7 * 2048 = 14336,  处理元素 [14336, 16384)
```

对应代码只需一行：

```python
offset = asc.get_block_idx() * block_length
```

### 2.1 查看Add样例中的SPMD代码

让我们查看Add样例的核函数代码，重点关注SPMD模型中`asc.get_block_idx()`的使用：

In [ ]:
# 查看Add样例核函数代码，重点关注SPMD并行相关接口
!cat ./src/add.py

在上面的代码中，核函数`vadd_kernel`的第一行就是：

```python
offset = asc.get_block_idx() * block_length
```

这行代码是SPMD模型的核心——每个核执行相同的代码，但通过不同的`block_idx`处理不同的数据分片。

在Add样例中，核函数的SPMD执行流程如下：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">步骤</th>
      <th align="left">代码</th>
      <th align="left">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>1</td><td><code>offset = asc.get_block_idx() * block_length</code></td><td>根据核编号计算数据偏移</td></tr>
    <tr><td>2</td><td><code>x_gm.set_global_buffer(x + offset, block_length)</code></td><td>绑定本核负责的GM地址段</td></tr>
    <tr><td>3</td><td><code>asc.data_copy(x_local[...], x_gm[...], tile_length)</code></td><td>搬入本核的数据分片</td></tr>
    <tr><td>4</td><td><code>asc.add(z_local[...], x_local[...], y_local[...], ...)</code></td><td>执行矢量加法</td></tr>
    <tr><td>5</td><td><code>asc.data_copy(z_gm[...], z_local[...], tile_length)</code></td><td>搬出结果到GM</td></tr>
  </tbody>
</table>


---
# 3. @asc.jit装饰器

`@asc.jit`是pyasc最核心的装饰器，用于定义核函数和Device侧执行函数。被`@asc.jit`修饰的函数会被JIT编译器处理，其Python代码将被转换为NPU可执行的Kernel二进制。

### 3.1 核函数 vs Device侧函数

被`@asc.jit`修饰的函数分为两类：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">类型</th>
      <th align="left">定义</th>
      <th align="left">调用方式</th>
      <th align="left">能否return</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>核函数（Kernel）</td>
      <td>由Host侧发起调用的函数</td>
      <td><code>kernel[核数, stream](args)</code></td>
      <td>不能</td>
    </tr>
    <tr>
      <td>Device侧函数</td>
      <td>被核函数或其他Device函数调用的函数</td>
      <td><code>func(args)</code> 直接调用</td>
      <td>可以</td>
    </tr>
  </tbody>
</table>

### 3.2 编译参数

通过`@asc.jit()`的小括号可以传入编译参数：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">参数名</th>
      <th align="left">含义</th>
      <th align="left">取值</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>kernel_type</code></td><td>核函数类型</td><td><code>KernelType</code>枚举</td></tr>
    <tr><td><code>opt_level</code></td><td>毕昇编译器优化级别</td><td>0, 1, 2, 3</td></tr>
    <tr><td><code>matmul_cube_only</code></td><td>是否纯Cube模式</td><td>True/False</td></tr>
    <tr><td><code>always_compile</code></td><td>是否强制重新编译</td><td>True/False</td></tr>
  </tbody>
</table>

### 3.3 运行参数

执行核函数时通过中括号`[]`传入运行时配置：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">参数</th>
      <th align="left">含义</th>
      <th align="left">是否必选</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>core_num</code></td><td>运行核数（不超过硬件可用核数）</td><td>是</td></tr>
    <tr><td><code>stream</code></td><td>Kernel执行流</td><td>否（默认当前流）</td></tr>
  </tbody>
</table>

调用示例：

```python
vadd_kernel[8, rt.current_stream()](x, y, z, block_length)
```


---
# 4. pyasc编译流程

pyasc的核心价值在于其自动编译链路，将Python源码逐步转换为NPU可执行Kernel。完整的编译流程如下：

```text
Python源码
  → AST（抽象语法树）
  → ASC-IR（基于MLIR的中间表示）
  → Ascend C代码
  → 毕昇编译器
  → NPU Kernel二进制
```

下图直观展示了pyasc的完整编译链路，以及前端模块和后端模块的分界：

<img src="./images/compilation_pipeline.png" alt="pyasc编译链路" width="800px">

前端模块（编译运行、Python前端、AST转ASC-IR）负责将Python代码转换为ASC-IR；后端模块（ASC-IR定义、Ascend C代码生成）负责将ASC-IR转换为最终可执行的Kernel。各模块的详细说明请参考[pyasc架构文档](https://gitcode.com/cann/pyasc/blob/master/docs/architecture_introduction.md)。

### 4.1 pyasc与Ascend C API命名对比

pyasc的API设计与Ascend C高度对齐，开发者如果已熟悉Ascend C，可以快速映射到pyasc：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">功能</th>
      <th align="left">Ascend C</th>
      <th align="left">pyasc</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>核函数定义</td><td><code>__global__ __aicore__</code></td><td><code>@asc.jit</code></td></tr>
    <tr><td>全局内存指针</td><td><code>__gm__ T*</code></td><td><code>asc.GlobalAddress</code></td></tr>
    <tr><td>数据搬运</td><td><code>DataCopy()</code></td><td><code>asc.data_copy()</code></td></tr>
    <tr><td>矢量加法</td><td><code>Add()</code></td><td><code>asc.add()</code></td></tr>
    <tr><td>同步事件</td><td><code>SetFlag()/WaitFlag()</code></td><td><code>asc.set_flag()/wait_flag()</code></td></tr>
    <tr><td>核函数调用</td><td><code>kernel<<<...>>>()</code></td><td><code>kernel[num, stream]()</code></td></tr>
  </tbody>
</table>


---
# 5. 小结

本节介绍了pyasc的核心编程范式：

- **Host-Device异构协同**：Host侧负责数据准备和核函数下发，Device侧负责并行计算
- **SPMD并行模型**：所有核执行同一核函数，通过`asc.get_block_idx()`处理不同数据分片
- **`@asc.jit`装饰器**：定义核函数和Device函数，编译参数通过`()`传入，运行参数通过`[]`传入
- **编译链路**：Python → AST → ASC-IR → Ascend C → Kernel二进制


---

## 课后练习

**选择题：**

1. pyasc采用什么并行编程模型？
   - A. SIMD
   - B. SPMD
   - C. MIMD
   - D. SISD

2. 在pyasc中，如何获取当前核的索引？
   - A. `asc.get_block_id()`
   - B. `asc.get_block_idx()`
   - C. `asc.get_core_id()`
   - D. `asc.get_core_idx()`

3. 以下哪个参数通过`@asc.jit()`小括号传入？
   - A. core_num
   - B. stream
   - C. always_compile
   - D. block_length

4. 以下哪个参数通过`kernel[]`中括号传入？
   - A. opt_level
   - B. core_num
   - C. matmul_cube_only
   - D. always_compile

<details>
<summary>点击查看答案</summary>

1. B  2. B  3. C  4. B

</details>